# TinyCast: training

**This is not the released run.** The released 146,505-parameter checkpoint is
36,621 optimizer steps at an effective batch of 4096 (128 per GPU, eight GPUs,
four gradient-accumulation steps), bf16-mixed, over GIFT-Eval-Pretrain and
Chronos KernelSynth plus four synthetic shards, and its weights are the uniform
mean of the last eight periodic checkpoints. None of that is a notebook
workload. What runs below is the same architecture and the same objective,
single-process, on synthetic data generated in this notebook, for a few hundred
steps at most.

`max_steps` reshapes a run rather than truncating one: warmup, stable and decay
are fractions of it and the scheduled-sampling probability ramps over its first
half, so a short run is not the prefix of a long one and their intermediate
states are not comparable.

Like the inference notebook, this one is both a starting point and a test. Copy
it and delete the assertion block at the end of each cell; run it unchanged and
the assertions say whether the objective, the rollout and the export path still
behave as the paper describes.

**Scale.** `TINYCAST_NB_SCALE` sets the number of optimizer steps: `smoke` (the
default) is a dozen and puts the whole notebook inside a minute on a CPU, `full`
is a few hundred. Both settings run the same lines; only the step count differs,
and the corpus is sized from it.

**Requirements.**
`pip install git+https://github.com/raws-labs/tinycast.git`
and nothing else. No network, no
credentials, no benchmark data, no GPU.

In [ ]:
import os

SCALE = os.environ.get("TINYCAST_NB_SCALE", "smoke").strip().lower()
STEPS = {"smoke": 12, "full": 300}.get(SCALE)

BATCH_SIZE = 4
DEVICE = "cpu"
SEED = 42
for var in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
            "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(var, "4")

print(f"scale={SCALE!r}, steps={STEPS}, batch_size={BATCH_SIZE}, device={DEVICE!r}")

assert STEPS is not None, f"TINYCAST_NB_SCALE must be 'smoke' or 'full', got {SCALE!r}"

## 1. The rollout contract

A training sample is one window of `L + K * p` values: the encoder context `L`,
then `K` autoregressive blocks of `p` steps each. Every block re-normalizes a
context that has slid forward by `p`, so the last block still needs a full `L`
of history behind it, which is why the window is that wide and not `L + p`.
`training_window_width` is the arithmetic; call it instead of hard-coding 2240.

A window of any other width is refused rather than padded or cropped, because a
silently reshaped window would train a different model.

In [ ]:
import tempfile
from pathlib import Path

import torch

from tinycast import (TinyCastConfig, TinyCastForPrediction, train,
                      training_window_width)
from tinycast.train import AR_CHUNKS

config = TinyCastConfig()
WIDTH = training_window_width(config)
n_parameters = sum(p.numel() for p in TinyCastForPrediction(config).parameters())

refusal = ""
with tempfile.TemporaryDirectory() as scratch:
    try:
        train(config, data=[{"window": torch.zeros(WIDTH - 1)}], max_steps=1,
              output_dir=scratch, batch_size=1, device=DEVICE)
    except ValueError as exc:
        refusal = str(exc)

print(f"L={config.seq_len}, K={AR_CHUNKS}, p={config.output_token_len} "
      f"-> window width {WIDTH}")
print(f"{n_parameters:,} parameters from the shipped config")
print(f"a {WIDTH - 1}-wide window is "
      f"{'refused' if refusal else 'NOT refused'}")
if refusal:
    print(f"  {refusal.splitlines()[0]}")

assert WIDTH == config.seq_len + AR_CHUNKS * config.output_token_len
assert WIDTH == 2240
assert n_parameters == 146_505
assert str(WIDTH) in refusal


## 2. Synthetic training data, generated here

`tinycast.synth` holds the three generator families of the pretraining corpus:
Gaussian process, spike trains, and the trend-seasonal-impulse process. They are
plain NumPy and reach nothing outside the process.

The published corpus mixes them 70/15/15 with the Gaussian process in the
majority. It is left out here because Algorithm 1 samples by dense Cholesky at
the full series length, which is a GPU job: `tinycast.corpus` requires CUDA and
refuses to fall back, since the CPU path draws from a different generator and
would produce a different corpus under the same seed.

Series are z-normalized per series, which is what the corpus builder does on
write, and the whole corpus is sized so the run makes exactly one pass over it.

In [ ]:
import numpy as np

from tinycast import generate_spikes, generate_tsi
from tinycast.scale import seasonal_scale_factor

N_WINDOWS = BATCH_SIZE * STEPS            # exactly one pass, no repetition
n_spikes = N_WINDOWS // 2
raw = np.concatenate([
    generate_tsi(N_WINDOWS - n_spikes, WIDTH, seed=0),
    generate_spikes(n_spikes, WIDTH, seed=1),
])
raw = (raw - raw.mean(1, keepdims=True)) / np.maximum(raw.std(1, keepdims=True), 1e-6)

# The scale factor is metadata, not a hyperparameter: it tells the model how many
# samples fall in one canonical daily cycle, and the committing term's seasonal
# copy folds at the lag it implies.
SCALE_FACTOR = seasonal_scale_factor("H", "Energy")
windows = [{"window": torch.from_numpy(w.astype("float32")),
            "scale_factor": SCALE_FACTOR}
           for w in raw]

print(f"{len(windows)} windows of {WIDTH} values, scale_factor={SCALE_FACTOR}")
print(f"per-series mean {raw.mean(1).max():.2e}, std {raw.std(1).min():.4f}")

assert len(windows) == N_WINDOWS
assert windows[0]["window"].shape == (WIDTH,)
assert np.isfinite(raw).all()

## 3. The committing term

The base objective is the nine-quantile pinball loss. On top of it sits the
paper's own term, evaluated at the median alone and weighted by 0.3. Write `a`
for the seasonal copy (the last cycle of the context repeated at the lag the
sample's scale factor implies, `round(24 / s)` clipped to `[2, L/2]`), `m` for
the median forecast and `y` for the target over the observed positions. The term
is the mean of `relu(|m - y| - |a - y|)`, multiplied by a window-level gate that
is 1 only when the copy beats the median summed over the window.

Two mechanisms, and they are separable. The per-position hinge stops as soon as
the median is at least as close as the copy. The gate silences the whole window
when the median is already the better predictor overall. Three cases below pin
that down: a window where the copy is good, a window where the gate closes, and
the same window with the gate disabled, which is what isolates the gate from the
hinge.

**A recorded negative result.** White noise does not work as the "term is
silent" case. An untrained median is worse than a lagged copy on almost any
input, so the gate stays open. What closes it is a good median, which is what
case B builds. The copy is made bad instead by choosing a signal whose period
does not divide the metadata lag.

In [ ]:
import math

from tinycast import committing_loss, seasonal_copy_baseline
from tinycast.losses import BASE_SEASONALITY, COMMIT_WEIGHT

L_DEMO, H_DEMO, S_DEMO = 240, 48, 1.0
LAG = round(BASE_SEASONALITY / S_DEMO)         # hourly data: 24
t = torch.arange(L_DEMO + H_DEMO, dtype=torch.float32)


def split(period):
    wave = torch.sin(2 * math.pi * t / period).unsqueeze(0)
    return wave[:, :L_DEMO], wave[:, L_DEMO:]


# A. The signal's period is the lag, so the copy is exact, and the median is
#    flat, so the gate opens and every position contributes.
ctx_a, tgt_a = split(LAG)
copy_a = seasonal_copy_baseline(ctx_a, H_DEMO, S_DEMO)
term_a = committing_loss(torch.zeros_like(tgt_a), tgt_a, copy_a)

# B. Period 7 does not divide the lag, so the copy is out of phase and bad. The
#    median is the true continuation with three positions knocked out: those
#    positions are worse than the copy, the window as a whole is not.
ctx_b, tgt_b = split(7.0)
copy_b = seasonal_copy_baseline(ctx_b, H_DEMO, S_DEMO)
median_b = tgt_b.clone()
median_b[0, [5, 17, 40]] += 2.0
term_b = committing_loss(median_b, tgt_b, copy_b, gated=True)

# C. The same window with the gate off: what is left is the per-position hinge.
term_c = committing_loss(median_b, tgt_b, copy_b, gated=False)

unweighted = committing_loss(torch.zeros_like(tgt_a), tgt_a, copy_a, weight=1.0)
print(f"copy error, period {LAG}: {(copy_a - tgt_a).abs().max():.2e}"
      f"   period 7: {(copy_b - tgt_b).abs().mean():.4f}")
print(f"A  copy good, median flat, gated:   {term_a.item():.6f}")
print(f"B  median good overall, gated:      {term_b.item():.6f}")
print(f"C  same window, gate disabled:      {term_c.item():.6f}")

assert term_a.item() > 0.0
assert term_b.item() == 0.0
assert term_c.item() > 0.0
assert math.isclose(term_a.item(), COMMIT_WEIGHT * unweighted.item(), rel_tol=1e-6)

## 4. A masked future cannot reach the loss

Scheduled sampling replaces a block's true values with the model's own median
before the next block reads them. An unobserved position is always replaced,
whatever the sampling probability says, so a value the mask calls missing must
never enter the loss and must never enter the next block's context either.

The test masks everything past the first block's target, then replaces those
values with large noise. The step loss has to come out bitwise identical: not
close, identical. Anything else is leakage.

In [ ]:
L, P = config.seq_len, config.output_token_len
probe = torch.from_numpy(raw[:BATCH_SIZE].astype("float32"))
mask = torch.ones_like(probe)
mask[:, L + P:] = 0.0                       # only the first block is observed


def one_step(window):
    """One optimizer step on a fixed batch; returns the loss it recorded."""
    samples = [{"window": window[i], "mask": mask[i], "scale_factor": SCALE_FACTOR}
               for i in range(window.shape[0])]
    with tempfile.TemporaryDirectory() as scratch:
        return train(config, data=samples, max_steps=1, output_dir=scratch,
                     batch_size=window.shape[0], device=DEVICE, seed=SEED,
                     deterministic=True).losses[0]


corrupted = probe.clone()
corrupted[:, L + P:] = 50.0 * torch.randn(
    BATCH_SIZE, WIDTH - L - P, generator=torch.Generator().manual_seed(7)
)
observed_loss = one_step(probe)
corrupted_loss = one_step(corrupted)

print(f"masked positions per sample: {int((mask[0] < 0.5).sum())} of {WIDTH}")
print(f"loss on the clean window:     {observed_loss!r}")
print(f"loss with the future ruined:  {corrupted_loss!r}")

assert observed_loss == corrupted_loss
assert math.isfinite(observed_loss)

## 5. Determinism

`train` seeds Python, NumPy, torch, the sampler and the workers from its `seed`
argument, inside the call. `deterministic=True` additionally asks for
deterministic kernels for the duration of the call and restores the process
flags afterwards; an operation with no deterministic implementation warns rather
than raising, so it tightens reproducibility without ruling a device out.

Two runs of the same seed over the same data must agree step for step.

In [ ]:
def short_run(seed):
    samples = [{"window": torch.from_numpy(w.astype("float32")),
                "scale_factor": SCALE_FACTOR}
               for w in raw[:BATCH_SIZE]]
    with tempfile.TemporaryDirectory() as scratch:
        return train(config, data=samples, max_steps=2, output_dir=scratch,
                     batch_size=BATCH_SIZE, device=DEVICE, seed=seed,
                     deterministic=True)


first = short_run(SEED)
again = short_run(SEED)
other = short_run(SEED + 1)

print(f"seed {SEED}:     {first.losses}")
print(f"seed {SEED} again: {again.losses}")
print(f"seed {SEED + 1}:     {other.losses}")

assert first.losses == again.losses
assert first.learning_rates == again.learning_rates
assert first.losses != other.losses

## 6. Train

The recipe: a four-block autoregressive rollout under scheduled sampling, the
nine-quantile pinball loss plus the gated committing term, AdamW at peak
learning rate 3e-3, and a warmup-stable-decay schedule with 5% warmup and 35%
decay. Every constant is a module-level default in `tinycast.train` that an
attribute of the same name on the config overrides.

Periodic checkpoints are written but not averaged; section 7 does the averaging.
`checkpoint_every` is set so the run leaves at least the eight the release
recipe averages.

A run this short on a corpus this small is not a model. What the loss curve
shows is that the objective and the schedule are wired up, nothing more.

In [ ]:
from tinycast.export import RELEASE_AVERAGE_N

# Point this at a real directory to keep the run and its checkpoints.
OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="tinycast-run-"))
CHECKPOINT_EVERY = max(1, STEPS // RELEASE_AVERAGE_N)

result = train(
    config,
    data=windows,
    max_steps=STEPS,
    output_dir=str(OUTPUT_DIR),
    batch_size=BATCH_SIZE,
    device=DEVICE,
    seed=SEED,
    checkpoint_every=CHECKPOINT_EVERY,
)

third = max(1, STEPS // 3)
opening = sum(result.losses[:third]) / third
closing = sum(result.losses[-third:]) / third
print(f"{result.steps} steps, window width {result.window_width}, "
      f"{len(result.checkpoints)} periodic checkpoints")
print(f"loss: first {third} steps {opening:.4f} -> last {third} steps {closing:.4f}")
print(f"learning rate: {result.learning_rates[0]:.2e} peak "
      f"{max(result.learning_rates):.2e} final {result.learning_rates[-1]:.2e}")
print(f"final checkpoint: {result.checkpoint_path}")

assert result.steps == STEPS
assert result.window_width == WIDTH
assert len(result.losses) == STEPS
assert all(math.isfinite(loss) for loss in result.losses)
assert closing < opening
assert max(result.learning_rates) <= 3e-3
assert len(result.checkpoints) >= RELEASE_AVERAGE_N
assert Path(result.checkpoint_path).is_file()
assert (OUTPUT_DIR / "config.json").is_file()

## 7. Average, export, reload

The released weights are the uniform mean of the last eight periodic
checkpoints, so exporting the final checkpoint alone does not reproduce them.
Pass the checkpoints in training order: an eight-term float32 sum is not
associative, and reordering moves the result by about one ulp.

`safetensors` writes each shared storage once, so an averaged state dict has 121
entries and not the 177 a `state_dict()` reports. Loading it with
`strict=False` is correct rather than lax: the 56 names it reports as missing
are exactly the weight-tied FFN aliases, and assigning the canonical name has
already written every one of them. The assertion below pins that set, so a
genuinely missing tensor still fails.

`check_export_roundtrip` then exports, reloads, and verifies three things: that
the reloaded model's forecasts are bitwise identical, that the tie is restored as
object identity rather than as equal copies, and that the file holds 146,505
values against the 321,225 a naive tensor sum reports.

In [ ]:
from tinycast import average_checkpoints, check_export_roundtrip, load_checkpoint
from tinycast.export import tied_parameter_groups

averaged = average_checkpoints(result.checkpoints[-RELEASE_AVERAGE_N:])

release = TinyCastForPrediction(config)
missing, unexpected = release.load_state_dict(averaged, strict=False)
aliases = {name
           for group in tied_parameter_groups(release).values()
           for name in group}

report = check_export_roundtrip(release, expect_parameters=146_505)
reloaded, reloaded_config = load_checkpoint(result.checkpoint_path)

print(f"averaged {RELEASE_AVERAGE_N} checkpoints into {len(averaged)} tensors")
print(f"load_state_dict: {len(missing)} missing (all tied aliases), "
      f"{len(unexpected)} unexpected")
print(f"export: {report['num_tensors']} tensors, "
      f"{report['num_parameters']:,} parameters, "
      f"{report['num_tied_aliases']} tied aliases, "
      f"state_dict sum {report['state_dict_sum']:,}")

assert set(missing) == aliases
assert not unexpected
assert report["num_parameters"] == 146_505
assert report["state_dict_sum"] == 321_225
assert report["bitwise_identical"] is True
assert report["num_tensors"] == len(averaged)
assert sum(p.numel() for p in reloaded.parameters()) == 146_505
assert reloaded_config.to_dict() == config.to_dict()

What a run like this does reproduce: the rollout, the objective, the schedule
shape, the optimizer, and the release path from periodic checkpoints to a
`model.safetensors` plus `config.json` pair. What it does not reproduce is the
released checkpoint or any number in the paper. For that, see the recipe at the
top of this notebook and `python -m tinycast.eval` for the evaluation side.

The eight-GPU launcher the released checkpoint was produced with is not part of
this package: `train` is single-process.